In [1]:
import torch                   # PyTorch：深度学习核心库，提供张量运算和自动求导
import torch.nn.functional as F # F 模块：包含各种不带可学习参数的函数（如交叉熵、softmax等）
import matplotlib.pyplot as plt # matplotlib：绘图库，用于可视化训练过程和结果
# 让图表直接嵌入 notebook 显示，无需调用 plt.show()
%matplotlib inline              

In [2]:
# 读取姓名数据集，每行一个名字，存为列表
words= open('D:\\Vault-4\\Projects\\makemore\\names.txt', 'r').read().splitlines()
print(len(words))              # 数据集中总共有多少个名字（32033个）
print(max(len(w) for w in words)) # 最长的名字有多少个字符（15个）
print(words[:8])               # 预览前8个名字，直观感受数据长什么样

32033
15
['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']


In [3]:
# --- 构建字符 ↔ 索引的双向映射表 ---
chars= sorted(list(set(''.join(words)))) # 将所有名字拼接→提取不重复字符→排序，得到 a-z 共26个字母
stoi= {s:i+1 for i,s in enumerate(chars)} # 字符→索引：a=1, b=2, ..., z=26（预留0给特殊字符）
stoi['.']=0                               # '.' 作为序列的起始/结束标记，索引设为 0
itos= {i:s for s,i in stoi.items()}       # 索引→字符：反向映射，用于把模型输出转回可读字符
vocab_size= len(stoi)                     # 词表大小 = 27（26个字母 + 1个特殊标记'.'）
print(itos)
print(vocab_size)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}
27


In [5]:
# --- 构建训练/验证/测试数据集 ---
block_size= 3  # 上下文窗口大小：用前3个字符预测下一个字符

def build_dataset(words):
    """
    将名字列表转换为训练样本 (X, Y)。
    采用滑动窗口方式：每次取 block_size 个字符作为输入，下一个字符作为目标。
    例如 "emma" → 输入/目标对：[...] → e, [..e] → m, [.em] → m, [emm] → a, [mma] → .
    """
    X, Y= [], []
    for w in words:
        context= [0]*block_size       # 初始上下文全为 0（即 '...'），表示序列开头
        for ch in w+'.':              # 遍历名字的每个字符，末尾加 '.' 标记结束
            ix= stoi[ch]              # 当前字符转为索引
            X.append(context)         # 当前上下文作为输入
            Y.append(ix)             # 当前字符作为预测目标
            context= context[1:]+[ix] # 滑动窗口：丢弃最左边字符，右边加入新字符
    X= torch.tensor(X)               # 转为 PyTorch 张量，形状 (样本数, block_size)
    Y= torch.tensor(Y)               # 转为 PyTorch 张量，形状 (样本数,)
    print(X.shape, Y.shape)
    return X, Y

import random
random.seed(42)                       # 固定随机种子，确保每次运行划分结果一致
random.shuffle(words)                 # 打乱名字顺序，避免数据有序带来的偏差
n1= int(0.8*len(words))              # 80% 数据用于训练
n2= int(0.9*len(words))              # 10% 用于验证（dev），剩余 10% 用于测试

Xtr, Ytr= build_dataset(words[:n1])   # 训练集：(182625, 3) — 约18万个训练样本
Xdev, Ydev= build_dataset(words[n1:n2]) # 验证集：(22655, 3) — 用于调超参数
Xte, Yte= build_dataset(words[n2:])   # 测试集：(22866, 3) — 最终评估模型性能

torch.Size([182580, 3]) torch.Size([182580])
torch.Size([22767, 3]) torch.Size([22767])
torch.Size([22799, 3]) torch.Size([22799])


In [6]:
# --- 梯度检验工具函数 ---
# 用于对比手动计算的梯度（dt）与 PyTorch 自动求导的梯度（t.grad）
# 这是本节课的核心：手动推导反向传播，然后用 autograd 验证正确性
# 参数说明：
#   s  — 字符串标签，打印时标识当前在检查哪个变量（如 "logprobs", "W2"）
#   dt — 你手动推导计算出的梯度张量（d=derivative, t=tensor）
#   t  — 前向传播中的原始张量，t.grad 里存着 PyTorch autograd 自动算出的梯度
# 调用示例：cmp('logprobs', dlogprobs, logprobs)
def cmp(s, dt, t):
    # torch.all(布尔张量)：检查张量中是否"每一个"元素都为 True
    # dt==t.grad 先逐元素比较 → 得到布尔张量 → torch.all 判断是否全部相等（bit-for-bit 精确匹配）
    ex=torch.all(dt==t.grad).item()
    # torch.allclose(a, b)：允许微小浮点误差的近似比较
    # 判定条件：|a_i - b_i| ≤ atol(1e-8) + rtol(1e-5) × |b_i|
    # 手动梯度和 autograd 的计算路径不同，浮点舍入会导致极微小差异，
    # 所以 exact 可能为 False 但 approximate 为 True —— 这就说明推导正确
    app=torch.allclose(dt, t.grad)
    maxdiff=(dt-t.grad).abs().max().item()   # 最大绝对差值：量化两者之间的最大偏差
    # f-string 格式化输出：
    #   {s:15s}         — 变量名左对齐占15字符宽，对齐打印结果方便纵向对比
    #   {str(ex):5s}    — exact 结果占5字符宽（"True " 或 "False"）
    #   {str(app):5s}   — approximate 结果占5字符宽
    #   {maxdiff}       — 最大差值，浮点数原样输出
    # 输出示例：logprobs        | exact:True  | approximate:True  | maxdiff:0.0
    print(f'{s:15s} | exact:{str(ex):5s} | approximate:{str(app):5s} | maxdiff:{maxdiff}')

In [7]:
# --- 初始化网络参数 ---
n_embd= 10    # 字符嵌入维度：每个字符用一个 10 维向量表示
n_hidden= 64  # 隐藏层神经元数量

g=torch.Generator().manual_seed(2147483647) # 固定随机种子，确保参数初始化可复现

# 嵌入矩阵 C：形状 (27, 10)，每行是一个字符的嵌入向量
# 通过 C[索引] 查表即可将字符转为连续向量
C= torch.randn((vocab_size, n_embd),           generator=g)

# === 第一层（隐藏层）===
# W1 形状 (30, 64)：输入 = 3个字符 × 10维嵌入 = 30维 → 64个隐藏神经元
# Kaiming 初始化：乘以 (5/3)/sqrt(fan_in)
#   - 5/3 是 tanh 激活函数的增益系数（补偿 tanh 压缩梯度的效应）
#   - 除以 sqrt(fan_in=30) 保证输出方差稳定，避免梯度爆炸/消失
W1= torch.randn((block_size*n_embd, n_hidden), generator=g)*(5/3)/((block_size*n_embd)**0.5)
b1= torch.randn(n_hidden,                      generator=g)*0.01 # 偏置初始化为很小的值，接近0

# === 第二层（输出层）===
# W2 形状 (64, 27)：64个隐藏神经元 → 27个输出类别（每个字符一个logit）
W2= torch.randn((n_hidden, vocab_size),        generator=g)*0.1  # 乘 0.1 使初始 logits 较小
b2= torch.randn(vocab_size,                    generator=g)*0.1  # 初始 logits 接近均匀 → 初始 loss ≈ -ln(1/27) ≈ 3.30

# === BatchNorm 参数 ===
# bngain (γ)：缩放因子，初始化为 ~1.0，形状 (1, 64) 可在 hidden 维度上广播
# 这里用 randn*0.1+1.0 而非 torch.ones()，只是在 1.0 附近加了微小随机扰动（约0.7~1.3）
# 两种写法效果几乎一样——PyTorch 官方 nn.BatchNorm 默认就是 γ=ones, β=zeros
bngain=torch.randn((1, n_hidden))*0.1+1.0
# bnbias (β)：偏移因子，初始化为 ~0.0，让网络学习每个神经元的最佳偏移
bnbias=torch.randn((1, n_hidden))*0.1

parameters= [C, W1, b1, W2, b2, bngain, bnbias]
print(sum(p.nelement() for p in parameters)) # 总参数量：4137
for p in parameters:
    p.requires_grad= True  # 开启梯度追踪，反向传播时自动计算每个参数的梯度

4137


In [8]:
# --- 采样一个 mini-batch ---
batch_size= 32
n=batch_size  # n 在后续 BatchNorm 计算中代表批次大小，用于求均值和方差

# 从训练集中随机采样 32 个样本的索引（使用固定 generator 保证可复现）
ix= torch.randint(0, Xtr.shape[0], (batch_size,), generator=g)
Xb, Yb= Xtr[ix], Ytr[ix]  # 取出对应的输入和目标
# Xb 形状 (32, 3)：32个样本，每个样本是3个字符索引
# Yb 形状 (32,)：32个目标字符索引
print(Xb.shape, Yb.shape)

torch.Size([32, 3]) torch.Size([32])


In [12]:
Xb, vocab_size

(tensor([[ 0,  7, 21],
         [ 2,  5, 14],
         [ 0,  0, 15],
         [12,  1,  9],
         [26,  5, 14],
         [ 0,  1, 26],
         [ 0, 12, 21],
         [19,  1,  8],
         [18,  9,  1],
         [ 0,  0, 20],
         [ 3, 15, 12],
         [ 6,  1,  9],
         [ 2,  5,  3],
         [ 0,  0, 10],
         [ 9, 11,  9],
         [ 0,  1,  1],
         [ 5,  4,  5],
         [ 0,  0,  0],
         [ 0,  1,  2],
         [25, 19,  8],
         [19,  5, 16],
         [ 1,  4, 22],
         [20,  1, 14],
         [13,  9, 18],
         [ 0,  0,  0],
         [19,  9,  1],
         [ 9, 20,  1],
         [19, 20, 15],
         [ 0, 19,  1],
         [ 0,  0,  0],
         [ 1, 14, 14],
         [ 0, 20, 18]]),
 27)

In [9]:
# ============================================================
# 前向传播 —— 刻意拆分为细粒度的原子操作
# 目的：后续手动为每一步写反向传播梯度，并用 autograd 验证
# ============================================================

# --- 1. 嵌入层 ---
emb= C[Xb]                        # (32, 3, 10) 查表：每个字符索引 → 10维嵌入向量
embcat= emb.view(emb.shape[0], -1) # (32, 30) 将3个嵌入向量拼接成一个30维向量（展平）

# --- 2. 第一层线性变换（pre-BatchNorm）---
hprebn= embcat @ W1 + b1           # (32, 64) 线性变换：30维输入 → 64维隐藏层（未经BN和激活）

# --- 3. BatchNorm（手动拆解为原子操作）---
# 3a. 计算批次均值：对32个样本求平均，得到每个神经元的均值
bnmeani=1/n * hprebn.sum(0, keepdim=True)    # (1, 64) 批次均值 μ

# 3b. 中心化：每个样本减去批次均值
bndiff= hprebn - bnmeani                     # (32, 64) x - μ

# 3c. 计算方差：中心化值的平方再求均值
bndiff2= bndiff**2                           # (32, 64) (x - μ)²

# 注意：除以 (n-1) 是无偏方差估计（Bessel 校正），与 PyTorch 的 BatchNorm 一致
bnvar= 1/(n-1)* bndiff2.sum(0, keepdim=True) # (1, 64) 批次方差 σ²

# 3d. 方差的逆平方根（加 ε=1e-5 防止除零）
bnvar_inv= (bnvar+1e-5)**-0.5                # (1, 64) 即 1/√(σ² + ε)

# 3e. 标准化：得到均值为0、方差为1的分布
bnraw= bndiff * bnvar_inv                    # (32, 64) 标准化后的值 x̂

# 3f. 仿射变换：γ * x̂ + β（可学习的缩放和偏移）
hpreact= bngain * bnraw + bnbias             # (32, 64) BatchNorm 最终输出

# --- 4. 激活函数 ---
h= torch.tanh(hpreact)             # (32, 64) tanh 将值压缩到 (-1, 1)

# --- 5. 第二层线性变换（输出层）---
logits= h @ W2 + b2                # (32, 27) 每个样本输出27个 logits，对应27个字符

# --- 6. 交叉熵损失（手动拆解，等价于 F.cross_entropy）---
# 6a. 数值稳定性：减去每行最大值，防止 exp() 溢出（不影响 softmax 结果）
logits_maxes= logits.max(1, keepdim=True).values  # (32, 1) 每个样本的最大 logit
norm_logits= logits - logits_maxes                 # (32, 27) 平移后的 logits

# 6b. Softmax 的分子：exp(logit)
counts= norm_logits.exp()                          # (32, 27) 指数化

# 6c. Softmax 的分母：每行 exp 值之和
counts_sum= counts.sum(1, keepdim=True)            # (32, 1) Σ exp(logit)

# 6d. 求倒数（拆出来方便后续手动求导）
counts_sum_inv= counts_sum**-1                     # (32, 1) 1 / Σ exp(logit)

# 6e. 得到概率分布
probs= counts * counts_sum_inv                     # (32, 27) softmax 概率

# 6f. 取对数概率
logprobs=probs.log()                               # (32, 27) log(softmax)

# 6g. 负对数似然损失（NLL）：
# logprobs[range(n), Yb] 是高级索引（fancy indexing）：
#   range(n)=[0,1,...,31] 选行，Yb=[各样本的目标字符索引] 选列
#   → 从每行中精确取出"正确类别"的 log 概率，得到长度为 32 的一维张量
# .mean() 对 32 个样本求平均
# 前面的负号：因为 log(prob) < 0（概率<1），取负变正数；模型越准 → prob越大 → loss越小
# 整体等价于 F.cross_entropy(logits, Yb)
loss=-logprobs[range(n), Yb].mean()                # 标量，即交叉熵损失

# --- 7. 准备反向传播 ---
for p in parameters:
    p.grad= None  # 清空旧梯度（等同于 optimizer.zero_grad()）

# 对所有中间变量调用 retain_grad()：
# 默认情况下 PyTorch 只保留叶子节点（参数）的梯度，
# 中间变量的梯度在 backward() 后会被释放。
# retain_grad() 使它们保留，方便我们用 cmp() 对比验证手动梯度。
for t in [logprobs, probs, counts, counts_sum, counts_sum_inv,
          norm_logits, logits_maxes, logits, h, hpreact, bnraw,
          bnvar_inv, bnvar, bndiff2, bndiff, hprebn, bnmeani, embcat, emb]:
    t.retain_grad()

loss.backward()  # PyTorch 自动反向传播，计算所有 .grad
loss              # 输出损失值 ≈ 3.48

tensor(3.4932, grad_fn=<NegBackward0>)

In [17]:
dlogprobs= torch.zeros_like(logprobs)  # ([32,27]) 手动计算 logprobs 的梯度
dlogprobs[range(n), Yb]= -1/n 
cmp('logprobs', dlogprobs, logprobs)  # ([32,27])  验证 logprobs 的梯度

dlogprobs_probs=1/probs
dprobs= dlogprobs * dlogprobs_probs  # 链式法则：dL/dprobs = dL/dlogprobs * dlogprobs/dprobs
cmp('probs', dprobs, probs)  # ([32,27])  验证 probs 的梯度

dcounts_sum_inv= (dprobs * counts).sum(1, keepdim=True)  # 链式法则：dL/dcounts_sum_inv = Σ(dL/dprobs * dprobs/dcounts_sum_inv)
cmp('counts_sum_inv', dcounts_sum_inv, counts_sum_inv)  # ([32,1]) 验证 counts_sum_inv 的梯度

dcounts_sum=(-counts_sum**-2) * dcounts_sum_inv  # 链式法则：dL/dcounts_sum = dL/dcounts_sum_inv * dcounts_sum_inv/dcounts_sum
cmp('counts_sum', dcounts_sum, counts_sum)  # ([32,1])

dcounts_1= dcounts_sum*torch.ones_like(counts)  # 链式法则：dL/dcounts = dL/dcounts_sum * dcounts_sum/dcounts
dcounts_2= dprobs * counts_sum_inv  # 链式法则：dL/dcounts = dL/dprobs * dprobs/dcounts
dcounts= dcounts_1 + dcounts_2  # 同一变量的梯度需要累加
cmp('counts', dcounts, counts)  # ([32,27]) 验证 counts 的梯度

dnorm_logits= dcounts * counts  # 链式法则：dL/dnorm_logits = dL/dcounts * dcounts/dnorm_logits
cmp('norm_logits', dnorm_logits, norm_logits)  # ([32,27]) 验证 norm_logits 的梯度

dlogits_maxes=-1*dnorm_logits.sum(1, keepdim=True)  # 链式法则：dL/dlogits_maxes = Σ(dL/dnorm_logits * dnorm_logits/dlogits_maxes)
cmp('logits_maxes', dlogits_maxes, logits_maxes)  # ([32,1]) 验证 logits_maxes 的梯度

dlogits_1= 1*dnorm_logits  # 链式法则：dL/dlogits = dL/dnorm_logits * dnorm_logits/dlogits
max_indices=logits.max(1, keepdim=True).indices  # (32, 1) 每行最大值的索引
one_hot_max= torch.zeros_like(logits).scatter(1, max_indices, 1)  # (32, 27) one-hot 编码：最大值位置为1，其余为0
dlogits_2= dlogits_maxes * one_hot_max  # 链式法则：dL/dlogits = dL/dlogits_maxes * dlogits_maxes/dlogits
dlogits= dlogits_1 + dlogits_2  # 同一变量的梯度需要累加
cmp('logits', dlogits, logits)  # ([32,27]) 验证 logits 的梯度

dh= dlogits @ W2.t()
cmp('h', dh, h)  # ([32,64]) 验证 h 的梯度

dW2= h.t() @ dlogits
cmp('W2', dW2, W2)  # ([64,27])

db2= dlogits.sum(0)
cmp('b2', db2, b2)  # ([27,])

dhpreact= (1-h**2) * dh  # 链式法则：dL/dhpreact = dL/dh * dh/dhpreact，tanh 的导数是 1 - tanh²
cmp('hpreact', dhpreact, hpreact)  # ([32,64]) 验证 hpreact 的梯度

dbngain = (dhpreact * bnraw).sum(0, keepdim=True)  # 链式法则：dL/dbngain = Σ(dL/dhpreact * dhpreact/dbngain)，其中 dhpreact/dbngain = bnraw
cmp('bngain', dbngain, bngain)  # ([1,64]) 验证 bngain 的梯度

dbnraw= dhpreact * bngain  # 链式法则：dL/dbnraw = dL/dhpreact * dhpreact/dbnraw，其中 dhpreact/dbnraw = bngain
cmp('bnraw', dbnraw, bnraw)  # ([32,64]) 验证 bnraw 的梯度

dbnbias=1* dhpreact.sum(0, keepdim=True)  # 链式法则：dL/dbnbias = Σ(dL/dhpreact * dhpreact/dbnbias)，其中 dhpreact/dbnbias = 1
cmp('bnbias', dbnbias, bnbias)  # ([1,64]) 验证 bnbias 的梯度

dbnvar_inv= (dbnraw * bndiff).sum(0, keepdim=True)  # 链式法则：dL/dbnvar_inv = Σ(dL/dbnraw * dbnraw/dbnvar_inv)，其中 dbnraw/dbnvar_inv = bndiff
cmp('bnvar_inv', dbnvar_inv, bnvar_inv)  # ([1,64]) 验证 bnvar_inv 的梯度

dbnvar= -0.5 * (bnvar + 1e-5)**(-1.5) * dbnvar_inv  # 链式法则：dL/dbnvar = dL/dbnvar_inv * dbnvar_inv/dbnvar，其中 dbnvar_inv/dbnvar = -0.5 * (bnvar + ε)^(-1.5)
cmp('bnvar', dbnvar, bnvar)  # ([1,64]) 验证 bnvar 的梯度

dbndiff2= (1/(n-1)) * torch.ones_like(bndiff2) * dbnvar  # 链式法则：dL/dbndiff2 = dL/dbnvar * dbnvar/dbndiff2，其中 dbnvar/dbndiff2 = 1/(n-1)
cmp('bndiff2', dbndiff2, bndiff2)  # ([32,64]) 验证 bndiff2 的梯度

dbndiff_1=bnvar_inv * dbnraw  # 链式法则：dL/dbndiff = dL/dbnraw * dbnraw/dbndiff，其中 dbnraw/dbndiff = bnvar_inv
dbndiff_2= 2*bndiff * dbndiff2  # 链式法则：dL/dbndiff = dL/dbndiff2 * dbndiff2/dbndiff，其中 dbndiff2/dbndiff = 2*bndiff
dbndiff= dbndiff_1 + dbndiff_2  # 同一变量的梯度需要累加
cmp('bndiff', dbndiff, bndiff)  # ([32,64]) 验证 bndiff 的梯度

dbnmeani= -1*dbndiff.sum(0, keepdim=True)  # 链式法则：dL/dbnmeani = Σ(dL/dbndiff * dbndiff/dbnmeani)，其中 dbndiff/dbnmeani = -1
cmp('bnmeani', dbnmeani, bnmeani)  # ([1,64]) 验证 bnmeani 的梯度

dbnrebn_1= dbndiff  # 链式法则：dL/dbnrebn_1 = dL/dbndiff * dbndiff/dbnrebn_1，其中 dbndiff/dbnrebn_1 = 1
dbnrebn_2= (1/n) * torch.ones_like(hprebn) * dbnmeani  # 链式法则：dL/dbnrebn_2 = dL/dbnmeani * dbnmeani/dbnrebn，其中 dbnmeani/dbnrebn = (1/n)
dbnrebn= dbnrebn_1 + dbnrebn_2  # 同一变量的梯度需要累加
cmp('hprebn', dbnrebn, hprebn)  # ([32,64]) 验证 hprebn 的梯度

dembcat= dbnrebn @ W1.t()  # 链式法则：dL/dembcat = dL/dbnrebn * dbnrebn/dembcat，其中 dbnrebn/dembcat = W1^T
cmp('embcat', dembcat, embcat)  # ([32,30])

dW1= embcat.t() @ dbnrebn  # 链式法则：dL/dW1 = dL/dbnrebn * dbnrebn/dW1，其中 dbnrebn/dW1 = embcat^T
cmp('W1', dW1, W1)  # ([30,64])

db1= dbnrebn.sum(0)  # 链式法则：dL/db1 = Σ(dL/dbnrebn * dbnrebn/db1)，其中 dbnrebn/db1 = 1
cmp('b1', db1, b1)  # ([64,])

demb= dembcat.view(emb.shape)  # 将展平的梯度重新变回嵌入层的形状 (32, 3, 10)
cmp('emb', demb, emb)  # ([32,3,10])

dC = torch.zeros_like(C)  # (27, 10) 先初始化为全零，因为不是每个字符都出现在当前 batch 中
# index_add_(dim, index, source)：
#   对于第 i 份 source，把它加到 dC 的第 index[i] 行上
#   等价于：for i in range(96): dC[index[i]] += source[i]
#
# Xb.view(-1)：(32,3) → (96,)  展平为 96 个字符索引，告诉每份梯度该加回 C 的第几行
# demb.view(-1, n_embd)：(32,3,10) → (96,10)  展平为 96 个 10 维梯度向量，与索引一一对应
#
# 例如字符 '.'（索引0）在 batch 中出现了 5 次，
# 则 dC[0] = 这 5 个位置的梯度向量之和
dC.index_add_(0, Xb.view(-1), demb.view(-1, n_embd))
cmp('C', dC, C)  # (27, 10) 验证嵌入矩阵 C 的梯度


logprobs        | exact:True  | approximate:True  | maxdiff:0.0
probs           | exact:True  | approximate:True  | maxdiff:0.0
counts_sum_inv  | exact:True  | approximate:True  | maxdiff:0.0
counts_sum      | exact:True  | approximate:True  | maxdiff:0.0
counts          | exact:True  | approximate:True  | maxdiff:0.0
norm_logits     | exact:True  | approximate:True  | maxdiff:0.0
logits_maxes    | exact:True  | approximate:True  | maxdiff:0.0
logits          | exact:True  | approximate:True  | maxdiff:0.0
h               | exact:True  | approximate:True  | maxdiff:0.0
W2              | exact:True  | approximate:True  | maxdiff:0.0
b2              | exact:True  | approximate:True  | maxdiff:0.0
hpreact         | exact:True  | approximate:True  | maxdiff:0.0
bngain          | exact:True  | approximate:True  | maxdiff:0.0
bnraw           | exact:True  | approximate:True  | maxdiff:0.0
bnbias          | exact:True  | approximate:True  | maxdiff:0.0
bnvar_inv       | exact:True  | approxim